In [1]:
# path: scripts/high_load_generator_fixed.py
# Saves y as per-sequence ragged array (one (T,) label array per sequence), just like Normal.
import os, random, numpy as np, pandas as pd

NUM_SEQUENCES = 665
bp = "../../../data/simulation/"
CSV_OUTPUT_PATH = os.path.join(bp, "engine_high_load_data.csv")
X_OUTPUT_PATH   = os.path.join(bp, "engine_high_load_X.npy")
Y_OUTPUT_PATH   = os.path.join(bp, "engine_high_load_y.npy")  # ragged per-sequence labels

# You can keep your exact physics; below is a minimal placeholder consistent with Normal
RPM_MIN, RPM_MAX = 2500.0, 6000.0
EDGE = 150.0; RPM_EPS = 1e-6; TEMP_MARGIN = 0.05; EPS = 1e-6; NUDGE = 1e-4
EDGE_RPM_WOBBLE = 4.0; EDGE_TEMP_WOBBLE = 0.25

rand_open = lambda lo,hi: lo + (hi-lo)*random.random()

def bounded_step(rpm, d):
    lo, hi = RPM_MIN+RPM_EPS, RPM_MAX-RPM_EPS
    d = min(max(d, lo-rpm), hi-rpm)
    new = rpm + d
    if new <= lo+EDGE_RPM_WOBBLE: new = lo+EDGE_RPM_WOBBLE
    if new >= hi-EDGE_RPM_WOBBLE: new = hi-EDGE_RPM_WOBBLE
    return new, new-rpm

def sample_delta(state):
    if state == "accel": return random.uniform(40.0, 120.0)
    if state == "decel": return -random.uniform(40.0, 120.0)
    return random.uniform(-39.999999, 39.999999)

def pick_state(desired, rpm):
    lo, hi = RPM_MIN+RPM_EPS, RPM_MAX-RPM_EPS
    if rpm >= hi-EDGE: return "decel"
    if rpm <= lo+EDGE: return "accel"
    return desired

def temp_from_rpm(rpm):
    frac = (rpm-RPM_MIN)/(RPM_MAX-RPM_MIN)
    span = 25.0 - 2.0*TEMP_MARGIN
    return (105+TEMP_MARGIN) + span*frac

def simulate_sequence_30s():
    R = rand_open(RPM_MIN+RPM_EPS, RPM_MAX-RPM_EPS)
    rpm,temp,pres,vib,labels = [],[],[],[],[]
    steps_left = 30
    while steps_left>0:
        run_len = min(int(random.uniform(1, steps_left+1)), steps_left)
        t = int(random.uniform(0,3)); base = "decel" if t==0 else ("accel" if t==1 else "steady")
        for _ in range(run_len):
            state = pick_state(base, R)
            d = sample_delta(state)
            Rn, d_eff = bounded_step(R, d)
            T_base = temp_from_rpm(Rn)
            transient = (d_eff/180.0)*0.5
            head_low, head_high = (T_base-105), (115.0-T_base)
            transient = min(transient, head_high-EPS) if transient>0 else max(transient, -head_low+EPS)
            rem_low, rem_high = head_low+transient, head_high-transient
            jitter = min(max(0.0, min(rem_low, rem_high)-EPS), 0.25)
            T = T_base + transient + random.uniform(-jitter, jitter)
            if T <= 105+NUDGE: T = 105+NUDGE+random.uniform(0.0, EDGE_TEMP_WOBBLE)
            elif T >= 115.0-NUDGE: T = 115.0-NUDGE-random.uniform(0.0, EDGE_TEMP_WOBBLE)
            P = 0.9 + (Rn-3500)/9000.0 + random.uniform(-0.07, 0.07)
            V = 0.1 + (Rn-3500)/20000.0 + random.uniform(-0.04, 0.05)
            rpm.append(Rn); temp.append(T); pres.append(P); vib.append(V)
            labels.append(
                "HighLoad (accelerating)" if state=="accel" else (
                "HighLoad (decelerating)" if state=="decel" else "HighLoad (idle)"))
            R = Rn
        steps_left -= run_len
    X_seq = np.stack([temp,pres,rpm,vib], axis=1).astype(np.float32)
    return X_seq, labels  # list[str] length 30

# ===== generate, save CSV, save X and ragged Y (per-sequence) =====
rows=[]; X_list=[]; y_list=[]; feature_cols=['Temperature','Pressure','RPM','Vibration']
for seq_id in range(NUM_SEQUENCES):
    X_seq, lab_seq = simulate_sequence_30s()
    X_list.append(X_seq)
    y_list.append(np.array(lab_seq, dtype=object))
    for t in range(30):
        rows.append({'Sequence':seq_id,'Time':t+1,
                     'Temperature':float(X_seq[t,0]),'Pressure':float(X_seq[t,1]),
                     'RPM':float(X_seq[t,2]),'Vibration':float(X_seq[t,3]),
                     'State':lab_seq[t]})

df = pd.DataFrame(rows, columns=['Sequence','Time',*feature_cols,'State'])
X = np.stack(X_list, axis=0)          # (N,30,4)
y = np.array(y_list, dtype=object)    # (N,) object; each element length 30

os.makedirs(bp, exist_ok=True)
df.to_csv(CSV_OUTPUT_PATH, index=False)
np.save(X_OUTPUT_PATH, X)
np.save(Y_OUTPUT_PATH, y)
print(f"X: {X.shape} | y: {y.shape} | CSV rows: {len(df)}")


X: (665, 30, 4) | y: (665, 30) | CSV rows: 19950
